# End-to-End Explainable CKD Risk Prediction & Severity Assessment

## Complete ML Pipeline — Self-Contained Google Colab Notebook

**CSE Capstone Project**

This notebook runs the **entire CKD risk prediction pipeline** with all code inline (no external imports needed):

1. **Install Dependencies** & Upload Dataset
2. **Data Loading** (2,159 patients, 11 features)
3. **Preprocessing** (Encoding, Scaling, 70/20/10 Stratified Split, SMOTE)
4. **Model Training** (5 models: LightGBM, CatBoost, XGBoost, Random Forest, Stacking Ensemble)
5. **Evaluation** (Validation + Test metrics, ROC curves, Confusion Matrices)
6. **Risk Scoring** (Low / Medium / High)
7. **SHAP Explainability** (Global & Local feature explanations)
8. **eGFR & CKD Staging** (Stages 1–5 per KDIGO guidelines)
9. **Reporting** (Patient Report Card, Correlation Heatmap, Dashboard)

### Key Design Decisions
- **eGFR, serum creatinine, ckd_stage excluded** from prediction features (data leakage — eGFR defines CKD clinically; serum creatinine is used to compute eGFR)
- **SMOTE (0.6 ratio)** applied on training data only to handle class imbalance (94.8% CKD)
- **Clinical Priority**: Models selected by Recall first (minimize missed CKD cases), then F1-Score
- **Reproducibility**: Random seed = 42 throughout

---

## Step 0: Install Dependencies & Mount Google Drive

Run this cell first to install packages and connect to your Google Drive where the dataset is stored.

In [ ]:
# Install required packages (run once)
!pip install -q lightgbm catboost xgboost shap imbalanced-learn openpyxl

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Dataset path from Google Drive
DATA_PATH = "/content/drive/MyDrive/Capstone Dataset/final_verified_ckd_dataset_fixed.xlsx"

import os
if os.path.exists(DATA_PATH):
    print(f"Dataset found: {DATA_PATH}")
else:
    print("ERROR: Dataset not found! Check the path.")

In [ ]:
# ── Import All Libraries ──
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve,
)
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

np.random.seed(42)
RANDOM_STATE = 42

print("All libraries imported successfully!")

---
## Module 1: Data Loading

Load the dataset and perform initial inspection and cleaning.

In [ ]:
# ── Module 1: Load and Clean Data ──

# Columns to drop (consent/timestamp)
DROP_COLUMNS = [
    "i_agree_that_this_data_may_be_used_for_research_purposes",
    "timestamp",
]

# Display name mapping for plots
DISPLAY_NAMES = {
    "age": "Age", "gender": "Gender",
    "have_you_ever_been_diagnosed_with_diabetes": "Diabetes",
    "do_you_have_high_blood_pressure_hypertension": "Hypertension",
    "do_you_notice_foamy_urine": "Foamy Urine",
    "do_you_take_extra_salt_with_your_food": "Extra Salt Intake",
    "serum_creatinine": "Serum Creatinine",
    "egfr": "eGFR", "hemoglobin": "Hemoglobin",
    "ckd_diagnosis": "CKD Diagnosis", "ckd_stage": "CKD Stage",
}

def get_display_name(col):
    return DISPLAY_NAMES.get(col, col.replace("_", " ").title())

# Load dataset
df = pd.read_excel(DATA_PATH)
df.columns = [c.strip().lower() for c in df.columns]

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nNull values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

# Fill hemoglobin nulls with median
if df["hemoglobin"].isnull().any():
    median_hb = df["hemoglobin"].median()
    df["hemoglobin"] = df["hemoglobin"].fillna(median_hb)
    print(f"\nFilled hemoglobin nulls with median: {median_hb:.2f}")

# Drop irrelevant columns
cols_to_drop = [c for c in DROP_COLUMNS if c in df.columns]
df = df.drop(columns=cols_to_drop)
print(f"\nCleaned dataset shape: {df.shape}")

# Keep a copy of original data
df_original = df.copy()

# Summary statistics
total = len(df)
ckd_pos = int((df["ckd_diagnosis"] == 1).sum())
ckd_neg = int((df["ckd_diagnosis"] == 0).sum())
print(f"\n{'='*50}")
print(f"Total patients: {total}")
print(f"CKD positive:   {ckd_pos} ({round(ckd_pos/total*100, 1)}%)")
print(f"CKD negative:   {ckd_neg} ({round(ckd_neg/total*100, 1)}%)")
print(f"Average Age:    {df['age'].mean():.1f}")
print(f"Average eGFR:   {df['egfr'].mean():.1f}")
print(f"{'='*50}")

df.head(10)

In [ ]:
# ── Data Exploration: Target Distribution ──

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
df['ckd_diagnosis'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('CKD Diagnosis Distribution')
axes[0].set_xlabel('Diagnosis (0=No CKD, 1=CKD)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Not CKD', 'CKD'], rotation=0)

# Descriptive statistics
df.describe().T.style.background_gradient(cmap='Blues')
axes[1].axis('off')
axes[1].text(0.1, 0.5, f"CKD: {ckd_pos} ({round(ckd_pos/total*100,1)}%)\n"
             f"Not CKD: {ckd_neg} ({round(ckd_neg/total*100,1)}%)\n\n"
             f"Total: {total} patients\n"
             f"Features: {len(df.columns)-1}",
             fontsize=14, verticalalignment='center', fontfamily='monospace')
axes[1].set_title('Summary')

plt.tight_layout()
plt.show()

print("\nDescriptive Statistics:")
df.describe()

---
## Module 2: Preprocessing

- Encode categorical features (Yes/No → 1/0, Male/Female → 1/0)
- **Exclude** eGFR, serum_creatinine, ckd_stage from prediction (data leakage)
- 70% Train / 20% Validation / 10% Test **stratified split**
- Scale numerical features (Age, Hemoglobin) using StandardScaler
- Apply **SMOTE** (sampling_strategy=0.6) on training data only

In [ ]:
# ── Module 2: Preprocessing ──

# Configuration
BINARY_YES_NO_COLUMNS = [
    "have_you_ever_been_diagnosed_with_diabetes",
    "do_you_have_high_blood_pressure_hypertension",
    "do_you_notice_foamy_urine",
    "do_you_take_extra_salt_with_your_food",
]
NUMERICAL_FEATURES = ["age", "hemoglobin"]
EXCLUDE_FROM_PREDICTION = ["egfr", "ckd_stage", "serum_creatinine"]
TARGET = "ckd_diagnosis"

# ── Step 2a: Encode Categorical Features ──
def encode_features(df):
    """Encode categorical columns to numerical values."""
    df = df.copy()
    for col in BINARY_YES_NO_COLUMNS:
        if col in df.columns:
            df[col] = df[col].map({"Yes": 1, "No": 0}).fillna(0).astype(int)
    if "gender" in df.columns:
        df["gender"] = df["gender"].map({"Male": 1, "Female": 0}).fillna(0).astype(int)
    return df

df_encoded = encode_features(df)

# ── Step 2b: Separate Features and Target ──
feature_cols = [c for c in df_encoded.columns if c != TARGET and c not in EXCLUDE_FROM_PREDICTION]
X = df_encoded[feature_cols]
y = df_encoded[TARGET]
feature_names = list(X.columns)

print(f"Prediction features ({len(feature_names)}): {feature_names}")
print(f"Excluded from prediction: {EXCLUDE_FROM_PREDICTION}")
print(f"\nTarget distribution:\n{y.value_counts()}")

# ── Step 2c: Stratified 70/20/10 Split ──
# Step 1: Split off 10% test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=RANDOM_STATE, stratify=y
)
# Step 2: Split remaining 90% into ~70% train and ~20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2222, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"\n{'='*50}")
print(f"Stratified Split Results:")
print(f"  Training:   {len(X_train)} samples ({len(X_train)/len(X)*100:.0f}%)")
print(f"  Validation: {len(X_val)} samples ({len(X_val)/len(X)*100:.0f}%)")
print(f"  Test:       {len(X_test)} samples ({len(X_test)/len(X)*100:.0f}%)")
print(f"{'='*50}")

# Save validation indices for report generation
val_indices = np.array(X_val.index)

# ── Step 2d: Scale Numerical Features ──
scaler = StandardScaler()
num_cols = [c for c in NUMERICAL_FEATURES if c in X_train.columns]

X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"\nScaled numerical features: {num_cols}")
print(f"Scaler mean: {scaler.mean_}")
print(f"Scaler std:  {scaler.scale_}")

# ── Step 2e: Apply SMOTE on Training Data Only ──
print(f"\nBefore SMOTE — Train class distribution:")
print(y_train.value_counts())

smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.6)
X_train_values, y_train_values = smote.fit_resample(X_train, y_train)
X_train = pd.DataFrame(X_train_values, columns=feature_names)
y_train = pd.Series(y_train_values, name=TARGET)

print(f"\nAfter SMOTE (ratio=0.6) — Train class distribution:")
print(y_train.value_counts())
print(f"\nTotal training samples after SMOTE: {len(X_train)}")

---
## Module 3: Model Training

Train 5 ML models with tuned hyperparameters (300 epochs each, strong regularization):
- **LightGBM**: Gradient boosting with leaf-wise growth
- **CatBoost**: Gradient boosting with ordered boosting
- **XGBoost**: Gradient boosting with level-wise growth
- **Random Forest**: Bagging ensemble of decision trees
- **Stacking Ensemble**: LightGBM + CatBoost + XGBoost with Logistic Regression meta-learner

In [ ]:
# ── Module 3: Model Training (5 Models, 300 Epochs Each) ──

# Define base models with tuned hyperparameters
base_models = {
    "LightGBM": LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6, num_leaves=31,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=RANDOM_STATE, verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, random_seed=RANDOM_STATE, verbose=0,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1, random_state=RANDOM_STATE, eval_metric="logloss",
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_split=5,
        min_samples_leaf=2, random_state=RANDOM_STATE,
    ),
}

# Convert to numpy arrays for training
X_train_arr = X_train.values
X_val_arr = X_val.values
X_test_arr = X_test.values

# Train base models
trained_models = {}
for name, model in base_models.items():
    print(f"Training {name}...")
    model.fit(X_train_arr, y_train)
    trained_models[name] = model
    print(f"  ✓ {name} trained successfully")

# Train Stacking Ensemble (uses fresh base models with 5-fold CV)
print("\nTraining Stacking Ensemble...")
stacking_base = {
    "LightGBM": LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6, num_leaves=31,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=RANDOM_STATE, verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, random_seed=RANDOM_STATE, verbose=0,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1, random_state=RANDOM_STATE, eval_metric="logloss",
    ),
}
stacking = StackingClassifier(
    estimators=[
        ("lgbm", stacking_base["LightGBM"]),
        ("catboost", stacking_base["CatBoost"]),
        ("xgboost", stacking_base["XGBoost"]),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5, n_jobs=-1,
)
stacking.fit(X_train_arr, y_train)
trained_models["Stacking"] = stacking
print("  ✓ Stacking Ensemble trained successfully")

print(f"\n{'='*50}")
print(f"All {len(trained_models)} models trained: {list(trained_models.keys())}")
print(f"{'='*50}")

---
## Module 4: Evaluation

- Evaluate all models on **Validation Set** (for model selection)
- 5-Fold **Stratified Cross-Validation** (Recall)
- Select best model by **Recall first, then F1-Score** (clinical priority)
- Report final unbiased metrics on **Test Set**

In [ ]:
# ── Module 4a: Validation Set Evaluation ──

val_results = []
for name, model in trained_models.items():
    y_pred = model.predict(X_val_arr)
    y_proba = model.predict_proba(X_val_arr)[:, 1]
    val_results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_val, y_pred), 4),
        "Precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_val, y_pred, zero_division=0), 4),
        "F1-Score": round(f1_score(y_val, y_pred, zero_division=0), 4),
        "ROC-AUC": round(roc_auc_score(y_val, y_proba), 4),
    })

val_comparison_df = pd.DataFrame(val_results)

# 5-Fold Stratified Cross-Validation
print("Running 5-Fold Stratified Cross-Validation...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = []
for name, model in trained_models.items():
    scores = cross_val_score(model, X_train_arr, y_train, cv=skf, scoring="recall", n_jobs=-1)
    cv_results.append({
        "Model": name,
        "CV Recall Mean": round(scores.mean(), 4),
        "CV Recall Std": round(scores.std(), 4),
    })
    print(f"  {name} — CV Recall: {scores.mean():.4f} ± {scores.std():.4f}")

cv_df = pd.DataFrame(cv_results)
val_comparison_df = val_comparison_df.merge(cv_df, on="Model", how="left")

# Select best model: highest Recall, then F1
val_sorted = val_comparison_df.sort_values(by=["Recall", "F1-Score"], ascending=False)
best_name = val_sorted.iloc[0]["Model"]
best_model = trained_models[best_name]

print(f"\n{'='*60}")
print(f"  VALIDATION SET RESULTS (Model Selection)")
print(f"{'='*60}")
display(val_comparison_df)
print(f"\n  ★ Best Model (selected on validation): {best_name}")
print(f"{'='*60}")

In [ ]:
# ── Module 4b: Test Set Evaluation (Unbiased Final Metrics) ──

test_results = []
for name, model in trained_models.items():
    y_pred_t = model.predict(X_test_arr)
    y_proba_t = model.predict_proba(X_test_arr)[:, 1]
    test_results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred_t), 4),
        "Precision": round(precision_score(y_test, y_pred_t, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred_t, zero_division=0), 4),
        "F1-Score": round(f1_score(y_test, y_pred_t, zero_division=0), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_proba_t), 4),
    })

test_results_df = pd.DataFrame(test_results)

print(f"{'='*60}")
print(f"  TEST SET RESULTS (Unbiased Final Evaluation)")
print(f"{'='*60}")
display(test_results_df)
print(f"{'='*60}")

In [ ]:
# ── Module 4c: Visualization — Model Comparison Bar Chart ──

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, (df_plot, title) in zip(axes, [(val_comparison_df, "Validation Set"), (test_results_df, "Test Set")]):
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
    available = [m for m in metrics if m in df_plot.columns]
    x = np.arange(len(df_plot))
    width = 0.15
    for i, metric in enumerate(available):
        ax.bar(x + i * width, df_plot[metric], width, label=metric)
    ax.set_xlabel("Model")
    ax.set_ylabel("Score")
    ax.set_title(f"Model Performance ({title})")
    ax.set_xticks(x + width * (len(available) - 1) / 2)
    ax.set_xticklabels(df_plot["Model"], rotation=15, ha="right")
    ax.legend(loc="lower right", fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Module 4d: ROC Curves (Validation & Test) ──

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (X_eval, y_eval, title) in zip(axes, [
    (X_val_arr, y_val, "Validation Set"),
    (X_test_arr, y_test, "Test Set"),
]):
    for name, model in trained_models.items():
        y_proba = model.predict_proba(X_eval)[:, 1]
        fpr, tpr, _ = roc_curve(y_eval, y_proba)
        auc_val = roc_auc_score(y_eval, y_proba)
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves — {title}")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Module 4e: Confusion Matrices (Validation & Test) ──

for X_eval, y_eval, set_name in [(X_val_arr, y_val, "Validation"), (X_test_arr, y_test, "Test")]:
    fig, axes = plt.subplots(1, 5, figsize=(25, 4))
    for idx, (name, model) in enumerate(trained_models.items()):
        y_pred = model.predict(X_eval)
        cm = confusion_matrix(y_eval, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                    xticklabels=["Not CKD", "CKD"], yticklabels=["Not CKD", "CKD"])
        axes[idx].set_title(f"{name}")
        axes[idx].set_xlabel("Predicted")
        axes[idx].set_ylabel("Actual")
    plt.suptitle(f"Confusion Matrices ({set_name} Set)", fontsize=14, y=1.05)
    plt.tight_layout()
    plt.show()

---
## Module 5: Risk Scoring

Assign CKD risk categories based on prediction probability:
- **Low Risk**: Probability < 30%
- **Medium Risk**: 30% - 70%
- **High Risk**: > 70%

In [ ]:
# ── Module 5: Risk Scoring ──

def get_risk_category(prob):
    if prob < 0.3: return "Low"
    elif prob < 0.7: return "Medium"
    else: return "High"

def get_risk_color(cat):
    return {"Low": "#2ecc71", "Medium": "#f39c12", "High": "#e74c3c"}.get(cat, "#95a5a6")

# Compute risk scores on validation set
probabilities = best_model.predict_proba(X_val_arr)[:, 1]
risk_df = pd.DataFrame({
    "Patient_Index": range(len(probabilities)),
    "CKD_Probability": np.round(probabilities, 4),
    "Risk_Category": [get_risk_category(p) for p in probabilities],
    "Actual_Label": y_val.values,
})

print("Risk Distribution (Validation Set):")
print(risk_df["Risk_Category"].value_counts())

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pie chart
counts = risk_df["Risk_Category"].value_counts()
colors = [get_risk_color(c) for c in counts.index]
axes[0].pie(counts, labels=counts.index, autopct="%1.1f%%", colors=colors, startangle=90)
axes[0].set_title("Risk Distribution")

# Bar chart
counts_ordered = risk_df["Risk_Category"].value_counts().reindex(["Low", "Medium", "High"], fill_value=0)
bar_colors = [get_risk_color(c) for c in counts_ordered.index]
bars = axes[1].bar(counts_ordered.index, counts_ordered.values, color=bar_colors, edgecolor="black")
for bar, val in zip(bars, counts_ordered.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5, str(val), ha="center", fontsize=12)
axes[1].set_title("Risk Category Counts")
axes[1].grid(axis="y", alpha=0.3)

# Histogram
axes[2].hist(risk_df["CKD_Probability"], bins=20, color="#3498db", edgecolor="black", alpha=0.8)
axes[2].axvline(0.3, color="#f39c12", linestyle="--", linewidth=2, label="Low/Medium (0.3)")
axes[2].axvline(0.7, color="#e74c3c", linestyle="--", linewidth=2, label="Medium/High (0.7)")
axes[2].set_title("Probability Distribution")
axes[2].set_xlabel("CKD Probability")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

risk_df.head(10)

---
## Module 6: SHAP Explainability

SHAP (SHapley Additive exPlanations) provides both **global** and **local** feature importance:
- **Beeswarm plot**: Shows how each feature impacts predictions across all patients
- **Bar plot**: Mean absolute SHAP value per feature
- **Waterfall plot**: Explains individual patient predictions

> Note: TreeExplainer does not support StackingClassifier, so we use the best base model (LightGBM) for SHAP.

In [ ]:
# ── Module 6: SHAP Explainability ──

# Use best base model for SHAP if Stacking is selected
shap_model = best_model
shap_model_name = best_name
if best_name == "Stacking":
    base_df = val_comparison_df[val_comparison_df["Model"] != "Stacking"]
    base_sorted = base_df.sort_values(by=["Recall", "F1-Score"], ascending=False)
    shap_model_name = base_sorted.iloc[0]["Model"]
    shap_model = trained_models[shap_model_name]
    print(f"Using {shap_model_name} for SHAP (TreeExplainer does not support Stacking)")

# Compute SHAP values
print(f"Computing SHAP values with {shap_model_name}...")
explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_val_arr)
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # positive class

print(f"SHAP values shape: {shap_values.shape}")

# ── Global SHAP: Beeswarm Plot ──
print("\n--- SHAP Beeswarm Summary ---")
shap.summary_plot(shap_values, X_val_arr, feature_names=feature_names, show=True)

In [ ]:
# ── SHAP Bar Plot (Mean |SHAP|) ──
mean_abs = np.abs(shap_values).mean(axis=0)
sorted_idx = np.argsort(mean_abs)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(sorted_idx)), mean_abs[sorted_idx], color="#e74c3c", edgecolor="black")
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([get_display_name(feature_names[i]) for i in sorted_idx])
ax.set_xlabel("Mean |SHAP Value|")
ax.set_title("Feature Importance (Mean |SHAP|)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP Waterfall Plot — Patient 0 (Local Explanation) ──
patient_idx = 0
expected = explainer.expected_value
if isinstance(expected, (list, np.ndarray)):
    expected = expected[1] if len(expected) > 1 else expected[0]

explanation = shap.Explanation(
    values=shap_values[patient_idx],
    base_values=expected,
    data=X_val_arr[patient_idx],
    feature_names=feature_names,
)
print(f"--- SHAP Waterfall: Patient {patient_idx} ---")
shap.waterfall_plot(explanation, show=True)

# Print top explanations
print(f"\nTop feature explanations for Patient {patient_idx}:")
abs_shap = np.abs(shap_values[patient_idx])
top_indices = np.argsort(abs_shap)[::-1][:5]
for idx in top_indices:
    val = shap_values[patient_idx][idx]
    direction = "increases" if val > 0 else "decreases"
    print(f"  - {get_display_name(feature_names[idx])} {direction} CKD risk (SHAP = {val:+.4f})")

---
## Module 7: eGFR & CKD Staging

CKD stages based on eGFR values (KDIGO guidelines):
| Stage | eGFR Range | Description |
|-------|-----------|-------------|
| Stage 1 | >= 90 | Normal or High |
| Stage 2 | 60 - 89 | Mildly Decreased |
| Stage 3a | 45 - 59 | Mild-Moderate Decrease |
| Stage 3b | 30 - 44 | Moderate-Severe Decrease |
| Stage 4 | 15 - 29 | Severely Decreased |
| Stage 5 | < 15 | Kidney Failure |

In [ ]:
# ── Module 7: eGFR & CKD Staging ──

def classify_egfr_stage(egfr):
    if egfr >= 90: return "Stage 1"
    elif egfr >= 60: return "Stage 2"
    elif egfr >= 45: return "Stage 3a"
    elif egfr >= 30: return "Stage 3b"
    elif egfr >= 15: return "Stage 4"
    else: return "Stage 5"

def get_stage_description(stage):
    descs = {
        "Stage 1": "Normal or High", "Stage 2": "Mildly Decreased",
        "Stage 3a": "Mild-Moderate Decrease", "Stage 3b": "Moderate-Severe Decrease",
        "Stage 4": "Severely Decreased", "Stage 5": "Kidney Failure",
    }
    return descs.get(stage, "Unknown")

def get_stage_color(stage):
    colors = {
        "Stage 1": "#2ecc71", "Stage 2": "#27ae60", "Stage 3a": "#f1c40f",
        "Stage 3b": "#e67e22", "Stage 4": "#e74c3c", "Stage 5": "#c0392b",
    }
    return colors.get(stage, "#95a5a6")


def estimate_egfr_ckd_epi(serum_creatinine, age, is_female):
    """Estimate eGFR using the CKD-EPI 2021 formula (no race coefficient).

    Formula:
      Male   (S.cr <= 0.9): eGFR = 142 * (S.cr/0.9)^(-0.302) * (0.9938)^Age
      Male   (S.cr >  0.9): eGFR = 142 * (S.cr/0.9)^(-1.200) * (0.9938)^Age
      Female (S.cr <= 0.7): eGFR = 142 * (S.cr/0.7)^(-0.241) * (0.9938)^Age
      Female (S.cr >  0.7): eGFR = 142 * (S.cr/0.7)^(-1.200) * (0.9938)^Age
    """
    if is_female:
        kappa = 0.7
        alpha = -0.241
    else:
        kappa = 0.9
        alpha = -0.302

    scr_ratio = serum_creatinine / kappa
    exponent = alpha if serum_creatinine <= kappa else -1.200
    egfr = 142 * (scr_ratio ** exponent) * (0.9938 ** age)
    return round(egfr, 2)


# Quick test of the eGFR formula
print("=== eGFR Formula Test Cases ===")
test_cases = [
    ("Male,   age 45, S.cr 0.8", 0.8, 45, False),
    ("Male,   age 55, S.cr 1.5", 1.5, 55, False),
    ("Female, age 40, S.cr 0.6", 0.6, 40, True),
    ("Female, age 65, S.cr 1.2", 1.2, 65, True),
    ("Male,   age 70, S.cr 4.0 (severe)", 4.0, 70, False),
]
for label, scr, age, is_f in test_cases:
    egfr = estimate_egfr_ckd_epi(scr, age, is_f)
    stage = classify_egfr_stage(egfr)
    print(f"  {label:40s} -> eGFR = {egfr:>6.2f}  Stage: {stage}")

# Compute staging for all patients
staging_df = pd.DataFrame()
staging_df["eGFR"] = df_original["egfr"].values
staging_df["CKD_Stage"] = staging_df["eGFR"].apply(classify_egfr_stage)
staging_df["Stage_Description"] = staging_df["CKD_Stage"].apply(get_stage_description)

print("\nCKD Stage Distribution (Full Dataset):")
print(staging_df["CKD_Stage"].value_counts())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Stage distribution bar
stage_order = ["Stage 1", "Stage 2", "Stage 3a", "Stage 3b", "Stage 4", "Stage 5"]
stage_counts = staging_df["CKD_Stage"].value_counts().reindex(stage_order, fill_value=0)
stage_colors = [get_stage_color(s) for s in stage_counts.index]
bars = axes[0].bar(stage_counts.index, stage_counts.values, color=stage_colors, edgecolor="black")
for bar, val in zip(bars, stage_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()+5, str(val), ha="center", fontsize=11)
axes[0].set_title("CKD Stage Distribution (based on eGFR)")
axes[0].set_xlabel("CKD Stage")
axes[0].set_ylabel("Number of Patients")
axes[0].grid(axis="y", alpha=0.3)

# eGFR histogram
axes[1].hist(staging_df["eGFR"], bins=30, color="#3498db", edgecolor="black", alpha=0.8)
for val, label in [(90, "Stage 1/2"), (60, "2/3a"), (45, "3a/3b"), (30, "3b/4"), (15, "4/5")]:
    axes[1].axvline(val, color="red", linestyle="--", alpha=0.7)
    axes[1].text(val+1, axes[1].get_ylim()[1]*0.85, label, fontsize=8, rotation=90)
axes[1].set_title("eGFR Distribution")
axes[1].set_xlabel("eGFR (mL/min/1.73m²)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

staging_df.head(10)


---
## Module 8: Reporting & Visualization

- Correlation Heatmap
- Feature Importance (from best model)
- Dashboard Summary
- Sample Patient Report Card

In [ ]:
# ── Module 8a: Correlation Heatmap ──

df_enc_for_corr = df_encoded.drop(columns=EXCLUDE_FROM_PREDICTION, errors="ignore")
df_enc_numeric = df_enc_for_corr.select_dtypes(include=[np.number])
short = {c: get_display_name(c) for c in df_enc_numeric.columns}
corr = df_enc_numeric.rename(columns=short).corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, linewidths=0.5, annot_kws={"size": 9})
ax.set_title("Feature Correlation Heatmap", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Module 8b: Feature Importance (Best Model) ──

fi_model = shap_model if not hasattr(best_model, "feature_importances_") else best_model
if hasattr(fi_model, "feature_importances_"):
    importances = fi_model.feature_importances_
    sorted_idx_fi = np.argsort(importances)
    display_names = [get_display_name(feature_names[i]) for i in sorted_idx_fi]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(display_names, importances[sorted_idx_fi], color="#2980b9", edgecolor="black")
    ax.set_xlabel("Importance")
    ax.set_title(f"Feature Importance ({best_name})")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Model does not have feature_importances_ attribute.")

In [ ]:
# ── Module 8c: Dashboard Summary (4-Panel) ──

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Model comparison
ax = axes[0, 0]
metrics = ["Accuracy", "Recall", "F1-Score", "ROC-AUC"]
x = np.arange(len(val_comparison_df))
width = 0.2
for i, m in enumerate(metrics):
    ax.bar(x + i * width, val_comparison_df[m], width, label=m)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(val_comparison_df["Model"], rotation=15, ha="right", fontsize=9)
ax.set_title("Model Performance (Validation)")
ax.legend(fontsize=8)
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)

# Panel 2: Risk distribution
ax = axes[0, 1]
risk_counts = risk_df["Risk_Category"].value_counts().reindex(["Low", "Medium", "High"], fill_value=0)
r_colors = [get_risk_color(c) for c in risk_counts.index]
ax.pie(risk_counts, labels=risk_counts.index, autopct="%1.1f%%", colors=r_colors)
ax.set_title("Risk Distribution")

# Panel 3: CKD stage distribution
ax = axes[1, 0]
s_counts = staging_df["CKD_Stage"].value_counts().reindex(stage_order, fill_value=0)
s_colors = [get_stage_color(s) for s in s_counts.index]
ax.bar(s_counts.index, s_counts.values, color=s_colors, edgecolor="black")
ax.set_title("CKD Stage Distribution")
ax.set_xlabel("Stage")
ax.set_ylabel("Count")
ax.grid(axis="y", alpha=0.3)

# Panel 4: Probability histogram
ax = axes[1, 1]
ax.hist(risk_df["CKD_Probability"], bins=20, color="#3498db", edgecolor="black", alpha=0.8)
ax.axvline(0.3, color="#f39c12", linestyle="--", linewidth=2)
ax.axvline(0.7, color="#e74c3c", linestyle="--", linewidth=2)
ax.set_title("CKD Probability Distribution")
ax.set_xlabel("Probability")
ax.grid(alpha=0.3)

plt.suptitle("CKD Risk Prediction — Dashboard Summary", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Module 8d: Sample Patient Report Card ──

patient_idx = 0
original_idx = val_indices[patient_idx]
patient_row = df_original.iloc[original_idx]

# Prediction
pred = int(best_model.predict(X_val_arr[patient_idx].reshape(1, -1))[0])
prob = float(best_model.predict_proba(X_val_arr[patient_idx].reshape(1, -1))[0, 1])
risk = get_risk_category(prob)
egfr_val = float(patient_row.get("egfr", 0))

# CKD Stage (only for CKD-positive predictions)
if pred == 1:
    stage = classify_egfr_stage(egfr_val)
    stage_desc = get_stage_description(stage)
else:
    stage = "N/A"
    stage_desc = "No CKD detected"

actual = "CKD" if int(y_val.iloc[patient_idx]) == 1 else "Not CKD"

# Print report
print("=" * 60)
print("       CKD RISK PREDICTION — PATIENT REPORT CARD")
print("=" * 60)
print(f"\n  Patient Index (Validation Set): {patient_idx}")
print(f"  Original Dataset Row:           {original_idx}")
print(f"\n--- DEMOGRAPHICS ---")
print(f"  Age:         {patient_row['age']}")
print(f"  Gender:      {patient_row['gender']}")
print(f"\n--- CLINICAL VALUES ---")
print(f"  Serum Creatinine: {patient_row['serum_creatinine']} mg/dL")
print(f"  eGFR:             {egfr_val} mL/min/1.73m²")
print(f"  Hemoglobin:       {patient_row['hemoglobin']} g/dL")
print(f"\n--- ML PREDICTION ---")
print(f"  Prediction:      {'CKD' if pred == 1 else 'Not CKD'}")
print(f"  CKD Probability: {prob*100:.2f}%")
print(f"  Risk Category:   {risk}")
print(f"\n--- CKD STAGING ---")
print(f"  eGFR:        {egfr_val} mL/min/1.73m²")
print(f"  CKD Stage:   {stage}")
print(f"  Description: {stage_desc}")
print(f"\n--- TOP EXPLANATIONS (SHAP) ---")
abs_shap = np.abs(shap_values[patient_idx])
top_idx = np.argsort(abs_shap)[::-1][:3]
for i, idx in enumerate(top_idx, 1):
    val = shap_values[patient_idx][idx]
    direction = "increases" if val > 0 else "decreases"
    print(f"  {i}. {get_display_name(feature_names[idx])} {direction} CKD risk (SHAP = {val:+.4f})")
print(f"\n  Actual Label: {actual}")
print(f"\n{'='*60}")
print("  DISCLAIMER: This tool is for decision support only.")
print("  Not a replacement for medical diagnosis.")
print("=" * 60)

---
## Pipeline Complete — Final Summary

### Results Summary

| Metric | Stacking | CatBoost | LightGBM | XGBoost | RandomForest |
|--------|----------|----------|----------|---------|-------------|
| **Selected on**: Validation set (Recall first, then F1-Score) |

### Pipeline Architecture
```
Raw Data (2159 patients, 12 columns)
    │
    ▼
Module 1: Data Loading & Cleaning
    │  ── Drop consent column, fill hemoglobin nulls
    ▼
Module 2: Preprocessing
    │  ── Encode categoricals, exclude eGFR/creatinine/stage
    │  ── 70/20/10 stratified split
    │  ── StandardScaler (age, hemoglobin)
    │  ── SMOTE (ratio=0.6, training only)
    ▼
Module 3: Model Training (5 models × 300 epochs)
    │  ── LightGBM, CatBoost, XGBoost, RandomForest, Stacking
    ▼
Module 4: Evaluation
    │  ── Validation metrics + 5-fold CV + Test metrics
    │  ── ROC curves, Confusion matrices
    ▼
Module 5: Risk Scoring (Low/Medium/High)
    ▼
Module 6: SHAP Explainability (Global + Local)
    ▼
Module 7: eGFR-based CKD Staging (Stages 1-5)
    ▼
Module 8: Reporting & Visualization
```

### Key Design Decisions
1. **eGFR excluded** from prediction — CKD is clinically defined by eGFR (data leakage)
2. **Serum creatinine excluded** — eGFR is computed from it via CKD-EPI formula
3. **SMOTE ratio 0.6** — Balances class imbalance without overfitting
4. **Recall prioritized** — Clinical priority to minimize missed CKD cases
5. **Stacking Ensemble** — Combines LightGBM + CatBoost + XGBoost with LogisticRegression meta-learner

---
## Module 9: Manual Patient Prediction with SHAP Explainability
Demonstrate end-to-end prediction for a manually entered patient — including auto-computed eGFR, model prediction, CKD stage, and SHAP explanations of WHY the model made that prediction. This mirrors the Manual Input mode in the Streamlit dashboard.

In [ ]:
# ── Module 9: Manual Patient Prediction with SHAP Explainability ──

# Step 1: Define a manual patient (you can change these values!)
manual_patient = {
    "age": 62,
    "gender": "Male",                     # "Male" or "Female"
    "hemoglobin": 9.5,                    # g/dL
    "serum_creatinine": 2.1,              # mg/dL
    "diabetes": "Yes",                    # "Yes" or "No"
    "hypertension": "Yes",
    "foamy_urine": "Yes",
    "extra_salt": "No",
}

print("=" * 60)
print("MANUAL PATIENT INPUT")
print("=" * 60)
for k, v in manual_patient.items():
    print(f"  {k:20s}: {v}")

# Step 2: Auto-compute eGFR using CKD-EPI 2021 formula
egfr_computed = estimate_egfr_ckd_epi(
    serum_creatinine=manual_patient["serum_creatinine"],
    age=manual_patient["age"],
    is_female=(manual_patient["gender"] == "Female"),
)
print(f"\nAuto-Computed eGFR (CKD-EPI 2021): {egfr_computed} mL/min/1.73m²")

# Step 3: Build feature vector matching training feature order
yes_no = lambda v: 1 if v == "Yes" else 0
input_data = {
    "age": manual_patient["age"],
    "gender": 1 if manual_patient["gender"] == "Male" else 0,
    "have_you_ever_been_diagnosed_with_diabetes": yes_no(manual_patient["diabetes"]),
    "do_you_have_high_blood_pressure_hypertension": yes_no(manual_patient["hypertension"]),
    "do_you_notice_foamy_urine": yes_no(manual_patient["foamy_urine"]),
    "do_you_take_extra_salt_with_your_food": yes_no(manual_patient["extra_salt"]),
    "hemoglobin": manual_patient["hemoglobin"],
}

# Build vector in correct feature order
input_vector = np.array(
    [input_data[f] for f in feature_names], dtype=float
).reshape(1, -1)

# Scale numerical features using the SAME scaler from training
num_cols = [c for c in NUMERICAL_FEATURES if c in feature_names]
num_indices = [feature_names.index(c) for c in num_cols]
num_vals = input_vector[0, num_indices].reshape(1, -1)
scaled_vals = scaler.transform(num_vals)
for i, idx in enumerate(num_indices):
    input_vector[0, idx] = scaled_vals[0, i]

# Step 4: Predict using best model
# (best_model and best_name from earlier modules)
pred = int(best_model.predict(input_vector)[0])
prob = float(best_model.predict_proba(input_vector)[0, 1])

# Risk category
def get_risk_category(p):
    if p < 0.30: return "Low"
    elif p < 0.70: return "Medium"
    else: return "High"

risk = get_risk_category(prob)

# Determine CKD stage (only if predicted CKD)
if pred == 1:
    ckd_stage = classify_egfr_stage(egfr_computed)
    stage_desc = get_stage_description(ckd_stage)
else:
    ckd_stage = "N/A"
    stage_desc = "No CKD detected"

print("\n" + "=" * 60)
print("PREDICTION RESULTS")
print("=" * 60)
print(f"  Prediction:        {'CKD' if pred == 1 else 'Not CKD'}")
print(f"  CKD Probability:   {prob*100:.1f}%")
print(f"  Risk Category:     {risk}")
print(f"  Computed eGFR:     {egfr_computed} mL/min/1.73m²")
print(f"  CKD Stage:         {ckd_stage} ({stage_desc})")

# Step 5: SHAP Explainability for this manual patient
# TreeExplainer doesn't support StackingClassifier — use best base model fallback
import shap as _shap

shap_model = best_model
shap_model_name = best_name
if best_name == "Stacking":
    base_models_dict = {k: v for k, v in trained_models.items() if k != "Stacking"}
    base_comparison = val_comparison_df[val_comparison_df["Model"] != "Stacking"].copy()
    base_comparison_sorted = base_comparison.sort_values(by=["Recall", "F1-Score"], ascending=False)
    shap_model_name = base_comparison_sorted.iloc[0]["Model"]
    shap_model = base_models_dict[shap_model_name]
    print(f"\nNote: Stacking is not supported by TreeExplainer.")
    print(f"      Using best base model for SHAP: {shap_model_name}")

manual_explainer = _shap.TreeExplainer(shap_model)
manual_shap = manual_explainer.shap_values(input_vector)
if isinstance(manual_shap, list):
    manual_shap = manual_shap[1]
patient_shap = manual_shap[0]

# Sort by absolute SHAP value
sorted_idx = np.argsort(np.abs(patient_shap))[::-1]

# SHAP bar chart
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(feature_names))[::-1]
shap_sorted = [patient_shap[i] for i in sorted_idx]
labels_sorted = [feature_names[i] for i in sorted_idx]
colors_sorted = ["#e74c3c" if v > 0 else "#2ecc71" for v in shap_sorted]

ax.barh(y_pos, shap_sorted, color=colors_sorted, edgecolor="black")
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_sorted)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("SHAP Value (Red = Increases CKD Risk, Green = Decreases)")
ax.set_title(f"SHAP Feature Impact for Manual Patient ({shap_model_name})")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# Top 3 text explanations
print("\n" + "=" * 60)
print("TOP 3 CONTRIBUTING FACTORS (SHAP)")
print("=" * 60)
readable_values = {
    "age": f"{manual_patient['age']} years",
    "gender": manual_patient["gender"],
    "hemoglobin": f"{manual_patient['hemoglobin']} g/dL",
    "have_you_ever_been_diagnosed_with_diabetes": manual_patient["diabetes"],
    "do_you_have_high_blood_pressure_hypertension": manual_patient["hypertension"],
    "do_you_notice_foamy_urine": manual_patient["foamy_urine"],
    "do_you_take_extra_salt_with_your_food": manual_patient["extra_salt"],
}
for rank, idx in enumerate(sorted_idx[:3], 1):
    fname = feature_names[idx]
    sv = patient_shap[idx]
    raw_val = readable_values.get(fname, "")
    direction = "INCREASES" if sv > 0 else "DECREASES"
    icon = "[+]" if sv > 0 else "[-]"
    print(f"  {icon} {rank}. {fname} = {raw_val}")
    print(f"      -> {direction} CKD risk (SHAP = {sv:+.4f})")

# Clinical guidance
print("\n" + "=" * 60)
print("CLINICAL GUIDANCE")
print("=" * 60)
if pred == 1:
    print(f"  CKD DETECTED — Stage: {ckd_stage} ({stage_desc})")
    print(f"  eGFR = {egfr_computed} mL/min/1.73m² | S.Cr = {manual_patient['serum_creatinine']} mg/dL")
    print("  Recommendation: Consider nephrology referral for further evaluation.")
    print("  Regular monitoring of kidney function (eGFR, serum creatinine) advised.")
    print("  Manage underlying conditions (diabetes, hypertension) if applicable.")
else:
    print(f"  NO CKD DETECTED — Probability {prob*100:.1f}% ({risk} risk)")
    print(f"  eGFR = {egfr_computed} mL/min/1.73m² | S.Cr = {manual_patient['serum_creatinine']} mg/dL")
    print("  Recommendation: Continue routine health check-ups annually.")
    print("  Maintain healthy lifestyle — stay hydrated, limit salt intake.")
print("\n  DISCLAIMER: This is decision support only — not a medical diagnosis.")


In [ ]:
# ── Final Summary Print ──

best_val = val_comparison_df[val_comparison_df["Model"] == best_name].iloc[0]
best_test = test_results_df[test_results_df["Model"] == best_name].iloc[0]

print("=" * 60)
print("  PIPELINE COMPLETE — FINAL SUMMARY")
print("=" * 60)
print(f"\n  Dataset:         {DATA_PATH}")
print(f"  Total patients:  {total}")
print(f"  Split:           70% Train / 20% Validation / 10% Test")
print(f"  Best Model:      {best_name}")
print(f"\n  Validation Metrics:")
print(f"    Accuracy:      {best_val['Accuracy']:.4f}")
print(f"    Recall:        {best_val['Recall']:.4f}")
print(f"    F1-Score:      {best_val['F1-Score']:.4f}")
print(f"    ROC-AUC:       {best_val['ROC-AUC']:.4f}")
print(f"\n  Test Metrics (unbiased):")
print(f"    Accuracy:      {best_test['Accuracy']:.4f}")
print(f"    Recall:        {best_test['Recall']:.4f}")
print(f"    F1-Score:      {best_test['F1-Score']:.4f}")
print(f"    ROC-AUC:       {best_test['ROC-AUC']:.4f}")
print(f"\n  Prediction Features ({len(feature_names)}): {feature_names}")
print(f"  Excluded (leakage): {EXCLUDE_FROM_PREDICTION}")
print(f"\n{'='*60}")